In [2]:
# Import libraries

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load final engineered features and target

PROJECT_PATH = "/content/drive/MyDrive/kickstarter-sucess-prediction (1)"
DATA_PATH = os.path.join(PROJECT_PATH, "data", "processed")

X = pd.read_csv(os.path.join(DATA_PATH, "X_encoded.csv"))
y = pd.read_csv(os.path.join(DATA_PATH, "y.csv")).squeeze()

print("Data loaded successfully!")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Data loaded successfully!
X shape: (331672, 219)
y shape: (331672,)

Target distribution:
target
0    197716
1    133956
Name: count, dtype: int64


In [4]:
# 80/20 stratified train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train-test split completed successfully!")

print("\nTraining features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(4))


Train-test split completed successfully!

Training features: (265337, 219)
Testing features: (66335, 219)

Training target distribution:
target
0    0.5961
1    0.4039
Name: proportion, dtype: float64

Testing target distribution:
target
0    0.5961
1    0.4039
Name: proportion, dtype: float64


In [5]:
# Train Logistic Regression

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=2000,
    solver="liblinear",
    random_state=42
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [6]:
# Evaluate Logistic Regression

y_pred_lr = logistic_model.predict(X_test)
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)

print("===== Logistic Regression Results =====")
print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1 Score : {lr_f1:.4f}")
print(f"ROC-AUC  : {lr_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))


===== Logistic Regression Results =====
Accuracy : 0.6122
Precision: 0.5699
Recall   : 0.1625
F1 Score : 0.2529
ROC-AUC  : 0.6500

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.92      0.74     39544
           1       0.57      0.16      0.25     26791

    accuracy                           0.61     66335
   macro avg       0.59      0.54      0.50     66335
weighted avg       0.60      0.61      0.54     66335


Confusion Matrix:
[[36259  3285]
 [22438  4353]]


In [7]:
# Train Random Forest - faster version

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [8]:
# Evaluate Random Forest

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test, y_prob_rf)

print("===== Random Forest Results =====")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

===== Random Forest Results =====
Accuracy : 0.6658
Precision: 0.5713
Recall   : 0.6914
F1 Score : 0.6257
ROC-AUC  : 0.7358

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.65      0.70     39544
           1       0.57      0.69      0.63     26791

    accuracy                           0.67     66335
   macro avg       0.66      0.67      0.66     66335
weighted avg       0.68      0.67      0.67     66335


Confusion Matrix:
[[25646 13898]
 [ 8268 18523]]


In [9]:
# Train XGBoost

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [10]:
# Evaluate XGBoost

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)
xgb_roc_auc = roc_auc_score(y_test, y_prob_xgb)

print("===== XGBoost Results =====")
print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")
print(f"ROC-AUC  : {xgb_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

===== XGBoost Results =====
Accuracy : 0.6918
Precision: 0.6475
Recall   : 0.5198
F1 Score : 0.5767
ROC-AUC  : 0.7549

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.81      0.76     39544
           1       0.65      0.52      0.58     26791

    accuracy                           0.69     66335
   macro avg       0.68      0.66      0.67     66335
weighted avg       0.69      0.69      0.68     66335


Confusion Matrix:
[[31962  7582]
 [12865 13926]]


In [11]:
# Train Gradient Boosting

from sklearn.ensemble import HistGradientBoostingClassifier

hgb_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

hgb_model.fit(X_train, y_train)

print("HistGradientBoosting trained successfully!")

HistGradientBoosting trained successfully!


In [12]:
# Evaluate HistGradientBoosting

y_pred_hgb = hgb_model.predict(X_test)
y_prob_hgb = hgb_model.predict_proba(X_test)[:, 1]

hgb_accuracy = accuracy_score(y_test, y_pred_hgb)
hgb_precision = precision_score(y_test, y_pred_hgb)
hgb_recall = recall_score(y_test, y_pred_hgb)
hgb_f1 = f1_score(y_test, y_pred_hgb)
hgb_roc_auc = roc_auc_score(y_test, y_prob_hgb)

print("===== HistGradientBoosting Results =====")
print(f"Accuracy : {hgb_accuracy:.4f}")
print(f"Precision: {hgb_precision:.4f}")
print(f"Recall   : {hgb_recall:.4f}")
print(f"F1 Score : {hgb_f1:.4f}")
print(f"ROC-AUC  : {hgb_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_hgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hgb))

===== HistGradientBoosting Results =====
Accuracy : 0.6962
Precision: 0.6436
Recall   : 0.5554
F1 Score : 0.5962
ROC-AUC  : 0.7599

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.79      0.76     39544
           1       0.64      0.56      0.60     26791

    accuracy                           0.70     66335
   macro avg       0.68      0.67      0.68     66335
weighted avg       0.69      0.70      0.69     66335


Confusion Matrix:
[[31303  8241]
 [11911 14880]]


In [13]:
# Model comparison

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "HistGradientBoosting"
    ],
    "Accuracy": [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy,
        hgb_accuracy
    ],
    "Precision": [
        lr_precision,
        rf_precision,
        xgb_precision,
        hgb_precision
    ],
    "Recall": [
        lr_recall,
        rf_recall,
        xgb_recall,
        hgb_recall
    ],
    "F1 Score": [
        lr_f1,
        rf_f1,
        xgb_f1,
        hgb_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        rf_roc_auc,
        xgb_roc_auc,
        hgb_roc_auc
    ]
})

print("===== MODEL COMPARISON =====")
display(model_comparison.round(4))

===== MODEL COMPARISON =====


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.6122,0.5699,0.1625,0.2529,0.6500
1,Random Forest,0.6658,0.5713,0.6914,0.6257,0.7358
2,XGBoost,0.6918,0.6475,0.5198,0.5767,0.7549
3,HistGradientBoosting,0.6962,0.6436,0.5554,0.5962,0.7599


In [14]:
# Soft Voting Ensemble using already-trained models

ensemble_prob = (
    0.15 * y_prob_lr +
    0.30 * y_prob_rf +
    0.30 * y_prob_xgb +
    0.25 * y_prob_hgb
)

ensemble_pred = (ensemble_prob >= 0.5).astype(int)

print("Ensemble predictions generated successfully!")

Ensemble predictions generated successfully!


In [15]:
# Evaluate Ensemble

ensemble_accuracy = accuracy_score(y_test, ensemble_pred)
ensemble_precision = precision_score(y_test, ensemble_pred)
ensemble_recall = recall_score(y_test, ensemble_pred)
ensemble_f1 = f1_score(y_test, ensemble_pred)
ensemble_roc_auc = roc_auc_score(y_test, ensemble_prob)

print("===== Ensemble Results =====")
print(f"Accuracy : {ensemble_accuracy:.4f}")
print(f"Precision: {ensemble_precision:.4f}")
print(f"Recall   : {ensemble_recall:.4f}")
print(f"F1 Score : {ensemble_f1:.4f}")
print(f"ROC-AUC  : {ensemble_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, ensemble_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, ensemble_pred))

===== Ensemble Results =====
Accuracy : 0.6935
Precision: 0.6397
Recall   : 0.5519
F1 Score : 0.5925
ROC-AUC  : 0.7544

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.79      0.75     39544
           1       0.64      0.55      0.59     26791

    accuracy                           0.69     66335
   macro avg       0.68      0.67      0.67     66335
weighted avg       0.69      0.69      0.69     66335


Confusion Matrix:
[[31217  8327]
 [12006 14785]]


In [16]:
# Final Model Comparison

final_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "HistGradientBoosting",
        "Weighted Ensemble"
    ],
    "Accuracy": [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy,
        hgb_accuracy,
        ensemble_accuracy
    ],
    "Precision": [
        lr_precision,
        rf_precision,
        xgb_precision,
        hgb_precision,
        ensemble_precision
    ],
    "Recall": [
        lr_recall,
        rf_recall,
        xgb_recall,
        hgb_recall,
        ensemble_recall
    ],
    "F1 Score": [
        lr_f1,
        rf_f1,
        xgb_f1,
        hgb_f1,
        ensemble_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        rf_roc_auc,
        xgb_roc_auc,
        hgb_roc_auc,
        ensemble_roc_auc
    ]
})

print("===== FINAL MODEL COMPARISON =====")
display(final_comparison.round(4))

===== FINAL MODEL COMPARISON =====


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.6122,0.5699,0.1625,0.2529,0.6500
1,Random Forest,0.6658,0.5713,0.6914,0.6257,0.7358
2,XGBoost,0.6918,0.6475,0.5198,0.5767,0.7549
3,HistGradientBoosting,0.6962,0.6436,0.5554,0.5962,0.7599
4,Weighted Ensemble,0.6935,0.6397,0.5519,0.5925,0.7544


In [17]:
# Tuned XGBoost - Version 2

from xgboost import XGBClassifier

xgb_tuned = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_tuned.fit(X_train, y_train)

print("Tuned XGBoost trained successfully!")

Tuned XGBoost trained successfully!


In [18]:
# Evaluate Tuned XGBoost

y_pred_xgb_tuned = xgb_tuned.predict(X_test)
y_prob_xgb_tuned = xgb_tuned.predict_proba(X_test)[:, 1]

xgb_tuned_accuracy = accuracy_score(y_test, y_pred_xgb_tuned)
xgb_tuned_precision = precision_score(y_test, y_pred_xgb_tuned)
xgb_tuned_recall = recall_score(y_test, y_pred_xgb_tuned)
xgb_tuned_f1 = f1_score(y_test, y_pred_xgb_tuned)
xgb_tuned_roc_auc = roc_auc_score(y_test, y_prob_xgb_tuned)

print("===== Tuned XGBoost Results =====")
print(f"Accuracy : {xgb_tuned_accuracy:.4f}")
print(f"Precision: {xgb_tuned_precision:.4f}")
print(f"Recall   : {xgb_tuned_recall:.4f}")
print(f"F1 Score : {xgb_tuned_f1:.4f}")
print(f"ROC-AUC  : {xgb_tuned_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_tuned))

===== Tuned XGBoost Results =====
Accuracy : 0.6983
Precision: 0.6503
Recall   : 0.5472
F1 Score : 0.5943
ROC-AUC  : 0.7622

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.80      0.76     39544
           1       0.65      0.55      0.59     26791

    accuracy                           0.70     66335
   macro avg       0.69      0.67      0.68     66335
weighted avg       0.69      0.70      0.69     66335



In [19]:
!pip install catboost -q

from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=100
)

cat_model.fit(X_train, y_train)

print("CatBoost trained successfully!")

0:	learn: 0.6448290	total: 166ms	remaining: 1m 22s
100:	learn: 0.6883887	total: 9.55s	remaining: 37.7s
200:	learn: 0.6960959	total: 18.4s	remaining: 27.4s
300:	learn: 0.7008747	total: 28.1s	remaining: 18.6s
400:	learn: 0.7048772	total: 35.7s	remaining: 8.81s
499:	learn: 0.7078922	total: 45.5s	remaining: 0us
CatBoost trained successfully!


In [20]:
# Evaluate CatBoost

y_pred_cat = cat_model.predict(X_test).astype(int).ravel()
y_prob_cat = cat_model.predict_proba(X_test)[:, 1]

cat_accuracy = accuracy_score(y_test, y_pred_cat)
cat_precision = precision_score(y_test, y_pred_cat)
cat_recall = recall_score(y_test, y_pred_cat)
cat_f1 = f1_score(y_test, y_pred_cat)
cat_roc_auc = roc_auc_score(y_test, y_prob_cat)

print("===== CatBoost Results =====")
print(f"Accuracy : {cat_accuracy:.4f}")
print(f"Precision: {cat_precision:.4f}")
print(f"Recall   : {cat_recall:.4f}")
print(f"F1 Score : {cat_f1:.4f}")
print(f"ROC-AUC  : {cat_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat))

===== CatBoost Results =====
Accuracy : 0.6980
Precision: 0.6513
Recall   : 0.5430
F1 Score : 0.5923
ROC-AUC  : 0.7631

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.80      0.76     39544
           1       0.65      0.54      0.59     26791

    accuracy                           0.70     66335
   macro avg       0.69      0.67      0.68     66335
weighted avg       0.69      0.70      0.69     66335



In [21]:
# Random Forest Feature Importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Top 30 Important Features:")
display(feature_importance.head(30))

Top 30 Important Features:


,Feature,Importance
0,goal,0.160548
1,usd_goal_real,0.158277
2,duration_days,0.098527
10,name_length,0.052622
11,name_word_count,0.049303
3,launch_year,0.043311
147,category_Tabletop Games,0.042629
7,deadline_year,0.033108
179,main_category_Music,0.023566
182,main_category_Technology,0.022860


In [22]:
# Threshold Tuning for XGBoost

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.05):

    y_pred_threshold = (y_prob_xgb_tuned >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred_threshold),
        "Precision": precision_score(y_test, y_pred_threshold),
        "Recall": recall_score(y_test, y_pred_threshold),
        "F1": f1_score(y_test, y_pred_threshold)
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))

,Threshold,Accuracy,Precision,Recall,F1
0,0.30,0.6371,0.5309,0.8709,0.6597
1,0.35,0.6652,0.5594,0.8056,0.6603
2,0.40,0.6848,0.5893,0.7244,0.6499
3,0.45,0.6963,0.6207,0.6378,0.6291
4,0.50,0.6983,0.6503,0.5472,0.5943
5,0.55,0.6928,0.6819,0.4486,0.5412
6,0.60,0.6794,0.7101,0.3484,0.4675
7,0.65,0.6643,0.7447,0.2569,0.3820
8,0.70,0.6467,0.7808,0.1740,0.2846


In [23]:
# Find threshold with highest accuracy

best_accuracy = threshold_df.loc[
    threshold_df["Accuracy"].idxmax()
]

print("Best threshold for accuracy:")
display(best_accuracy)

Best threshold for accuracy:


,4
Threshold,0.500000
Accuracy,0.698274
Precision,0.650284
Recall,0.547199
F1,0.594304


In [24]:
# Find threshold with highest F1

best_f1 = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

print("Best threshold for F1:")
display(best_f1)

Best threshold for F1:


,1
Threshold,0.350000
Accuracy,0.665245
Precision,0.559417
Recall,0.805644
F1,0.660324


In [25]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# Load original cleaned dataset for native CatBoost
import os

print(os.listdir('/content/drive/MyDrive'))

['Nanditha', 'GoogleDrive_ArvindRaoV', 'Classroom', 'Bank.pdf', 'InAmigos', 'UG_CSE(AI&ML)-2026_Curriculum&Syllabi_Draft.pdf', 'Doc', 'DBMS_LAB1.pdf', 'Copy_of_Decision_Tree_Exploratory_Lab_2_0.ipynb', 'Colab Notebooks', 'resume (3).pdf', 'kickstarter-sucess-prediction (1)', 'C3I POSTER (3).pdf']


In [27]:
import os

for item in os.listdir('/content/drive/MyDrive'):
    if 'kickstarter' in item.lower():
        print(item)

kickstarter-sucess-prediction (1)


In [28]:
# Train CatBoost using existing train/test data

from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=500,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

cat_model.fit(X_train, y_train)

print("CatBoost trained successfully!")

0:	total: 66.6ms	remaining: 33.2s
100:	total: 8.55s	remaining: 33.8s
200:	total: 15.1s	remaining: 22.4s
300:	total: 23.4s	remaining: 15.5s
400:	total: 29.7s	remaining: 7.32s
499:	total: 39.7s	remaining: 0us
CatBoost trained successfully!


In [29]:
# Evaluate CatBoost

y_pred_cat = cat_model.predict(X_test).astype(int).ravel()
y_prob_cat = cat_model.predict_proba(X_test)[:, 1]

cat_accuracy = accuracy_score(y_test, y_pred_cat)
cat_precision = precision_score(y_test, y_pred_cat)
cat_recall = recall_score(y_test, y_pred_cat)
cat_f1 = f1_score(y_test, y_pred_cat)
cat_roc_auc = roc_auc_score(y_test, y_prob_cat)

print("===== CatBoost Results =====")
print(f"Accuracy : {cat_accuracy:.4f}")
print(f"Precision: {cat_precision:.4f}")
print(f"Recall   : {cat_recall:.4f}")
print(f"F1 Score : {cat_f1:.4f}")
print(f"ROC-AUC  : {cat_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat))

===== CatBoost Results =====
Accuracy : 0.6975
Precision: 0.6527
Recall   : 0.5362
F1 Score : 0.5888
ROC-AUC  : 0.7614

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.81      0.76     39544
           1       0.65      0.54      0.59     26791

    accuracy                           0.70     66335
   macro avg       0.69      0.67      0.67     66335
weighted avg       0.69      0.70      0.69     66335



In [30]:
# Tune Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_tuned = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)

print("Tuned Random Forest trained successfully!")

Tuned Random Forest trained successfully!


In [31]:
# Evaluate Tuned Random Forest

y_pred_rf_tuned = rf_tuned.predict(X_test)
y_prob_rf_tuned = rf_tuned.predict_proba(X_test)[:, 1]

rf_tuned_accuracy = accuracy_score(y_test, y_pred_rf_tuned)
rf_tuned_precision = precision_score(y_test, y_pred_rf_tuned)
rf_tuned_recall = recall_score(y_test, y_pred_rf_tuned)
rf_tuned_f1 = f1_score(y_test, y_pred_rf_tuned)
rf_tuned_roc_auc = roc_auc_score(y_test, y_prob_rf_tuned)

print("===== Tuned Random Forest Results =====")
print(f"Accuracy : {rf_tuned_accuracy:.4f}")
print(f"Precision: {rf_tuned_precision:.4f}")
print(f"Recall   : {rf_tuned_recall:.4f}")
print(f"F1 Score : {rf_tuned_f1:.4f}")
print(f"ROC-AUC  : {rf_tuned_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tuned))

===== Tuned Random Forest Results =====
Accuracy : 0.6766
Precision: 0.5849
Recall   : 0.6865
F1 Score : 0.6316
ROC-AUC  : 0.7467

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.67      0.71     39544
           1       0.58      0.69      0.63     26791

    accuracy                           0.68     66335
   macro avg       0.67      0.68      0.67     66335
weighted avg       0.69      0.68      0.68     66335



In [32]:
# Current model comparison

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "HistGradientBoosting",
        "CatBoost",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy,
        hgb_accuracy,
        cat_accuracy,
        rf_tuned_accuracy
    ],
    "Precision": [
        lr_precision,
        rf_precision,
        xgb_precision,
        hgb_precision,
        cat_precision,
        rf_tuned_precision
    ],
    "Recall": [
        lr_recall,
        rf_recall,
        xgb_recall,
        hgb_recall,
        cat_recall,
        rf_tuned_recall
    ],
    "F1": [
        lr_f1,
        rf_f1,
        xgb_f1,
        hgb_f1,
        cat_f1,
        rf_tuned_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        rf_roc_auc,
        xgb_roc_auc,
        hgb_roc_auc,
        cat_roc_auc,
        rf_tuned_roc_auc
    ]
})

display(comparison.sort_values("Accuracy", ascending=False).round(4))

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
4,CatBoost,0.6975,0.6527,0.5362,0.5888,0.7614
3,HistGradientBoosting,0.6962,0.6436,0.5554,0.5962,0.7599
2,XGBoost,0.6918,0.6475,0.5198,0.5767,0.7549
5,Tuned Random Forest,0.6766,0.5849,0.6865,0.6316,0.7467
1,Random Forest,0.6658,0.5713,0.6914,0.6257,0.7358
0,Logistic Regression,0.6122,0.5699,0.1625,0.2529,0.6500


In [33]:
# ==========================================
# FEATURE SELECTION - TOP 50 FEATURES
# ==========================================

# Use Random Forest feature importance
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

# Select top 50
top_features = feature_importance.head(50)["Feature"].tolist()

print("Number of selected features:", len(top_features))
print("\nTop 50 features:")
print(top_features)

Number of selected features: 50

Top 50 features:
['goal', 'usd_goal_real', 'duration_days', 'name_length', 'name_word_count', 'launch_year', 'category_Tabletop Games', 'deadline_year', 'main_category_Music', 'main_category_Technology', 'main_category_Theater', 'category_Apps', 'main_category_Comics', 'category_Web', 'main_category_Food', 'launch_day', 'category_Shorts', 'category_Indie Rock', 'category_Hip-Hop', 'launch_weekday', 'main_category_Fashion', 'deadline_month', 'launch_month', 'category_Theater', 'main_category_Publishing', 'deadline_weekday', 'category_Country & Folk', 'main_category_Games', 'category_Video Games', 'category_Apparel', 'currency_EUR', 'category_Fiction', 'main_category_Film & Video', 'main_category_Crafts', 'currency_USD', 'country_US', 'category_Classical Music', 'main_category_Dance', 'category_Dance', 'category_Software', 'category_Fashion', 'category_Product Design', 'main_category_Design', 'category_Mobile Games', 'category_Comic Books', 'main_category

In [34]:
# Create datasets using only the selected features

X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

print("Original features :", X_train.shape[1])
print("Selected features:", X_train_top.shape[1])

print("\nTraining shape:", X_train_top.shape)
print("Testing shape :", X_test_top.shape)

Original features : 219
Selected features: 50

Training shape: (265337, 50)
Testing shape : (66335, 50)


In [35]:
# ==========================================
# CATBOOST WITH TOP 50 FEATURES
# ==========================================

cat_top50 = CatBoostClassifier(
    iterations=500,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

cat_top50.fit(X_train_top, y_train)

print("CatBoost Top-50 trained successfully!")

0:	total: 82.5ms	remaining: 41.2s
100:	total: 8.39s	remaining: 33.2s
200:	total: 14.8s	remaining: 21.9s
300:	total: 23.3s	remaining: 15.4s
400:	total: 29.8s	remaining: 7.35s
499:	total: 38.3s	remaining: 0us
CatBoost Top-50 trained successfully!


In [36]:
# ==========================================
# EVALUATE CATBOOST TOP 50
# ==========================================

y_pred_cat_top = cat_top50.predict(X_test_top).astype(int).ravel()
y_prob_cat_top = cat_top50.predict_proba(X_test_top)[:, 1]

cat_top_accuracy = accuracy_score(y_test, y_pred_cat_top)
cat_top_precision = precision_score(y_test, y_pred_cat_top)
cat_top_recall = recall_score(y_test, y_pred_cat_top)
cat_top_f1 = f1_score(y_test, y_pred_cat_top)
cat_top_roc_auc = roc_auc_score(y_test, y_prob_cat_top)

print("===== CatBoost Top-50 Results =====")
print(f"Accuracy : {cat_top_accuracy:.4f}")
print(f"Precision: {cat_top_precision:.4f}")
print(f"Recall   : {cat_top_recall:.4f}")
print(f"F1 Score : {cat_top_f1:.4f}")
print(f"ROC-AUC  : {cat_top_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat_top))

===== CatBoost Top-50 Results =====
Accuracy : 0.6937
Precision: 0.6462
Recall   : 0.5339
F1 Score : 0.5847
ROC-AUC  : 0.7565

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.80      0.76     39544
           1       0.65      0.53      0.58     26791

    accuracy                           0.69     66335
   macro avg       0.68      0.67      0.67     66335
weighted avg       0.69      0.69      0.69     66335



In [37]:
# ==========================================
# TUNED CATBOOST
# ==========================================

cat_tuned = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=5,
    random_strength=1,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200
)

cat_tuned.fit(X_train, y_train)

print("Tuned CatBoost trained successfully!")

0:	total: 80ms	remaining: 1m 19s
200:	total: 16.4s	remaining: 1m 5s
400:	total: 32.7s	remaining: 48.9s
600:	total: 48.6s	remaining: 32.3s
800:	total: 1m 6s	remaining: 16.6s
999:	total: 1m 23s	remaining: 0us
Tuned CatBoost trained successfully!


In [38]:
# ==========================================
# EVALUATE TUNED CATBOOST
# ==========================================

y_pred_cat_tuned = cat_tuned.predict(X_test).astype(int).ravel()
y_prob_cat_tuned = cat_tuned.predict_proba(X_test)[:, 1]

cat_tuned_accuracy = accuracy_score(y_test, y_pred_cat_tuned)
cat_tuned_precision = precision_score(y_test, y_pred_cat_tuned)
cat_tuned_recall = recall_score(y_test, y_pred_cat_tuned)
cat_tuned_f1 = f1_score(y_test, y_pred_cat_tuned)
cat_tuned_roc_auc = roc_auc_score(y_test, y_prob_cat_tuned)

print("===== Tuned CatBoost Results =====")
print(f"Accuracy : {cat_tuned_accuracy:.4f}")
print(f"Precision: {cat_tuned_precision:.4f}")
print(f"Recall   : {cat_tuned_recall:.4f}")
print(f"F1 Score : {cat_tuned_f1:.4f}")
print(f"ROC-AUC  : {cat_tuned_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_cat_tuned))

===== Tuned CatBoost Results =====
Accuracy : 0.6993
Precision: 0.6523
Recall   : 0.5471
F1 Score : 0.5951
ROC-AUC  : 0.7643

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.80      0.76     39544
           1       0.65      0.55      0.60     26791

    accuracy                           0.70     66335
   macro avg       0.69      0.67      0.68     66335
weighted avg       0.69      0.70      0.69     66335



In [39]:
# ==========================================
# ENSEMBLE: XGBOOST + CATBOOST + HISTGRADIENTBOOSTING
# ==========================================

# Probability predictions for class 1
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
cat_prob = cat_model.predict_proba(X_test)[:, 1]
hgb_prob = hgb_model.predict_proba(X_test)[:, 1]

print("Probabilities generated successfully!")

Probabilities generated successfully!


In [40]:
# ==========================================
# WEIGHTED SOFT VOTING
# ==========================================

ensemble_prob = (
    0.34 * xgb_prob +
    0.36 * cat_prob +
    0.30 * hgb_prob
)

ensemble_pred = (ensemble_prob >= 0.50).astype(int)

print("Ensemble predictions generated successfully!")

Ensemble predictions generated successfully!


In [41]:
# ==========================================
# EVALUATE ENSEMBLE
# ==========================================

ensemble_accuracy = accuracy_score(y_test, ensemble_pred)
ensemble_precision = precision_score(y_test, ensemble_pred)
ensemble_recall = recall_score(y_test, ensemble_pred)
ensemble_f1 = f1_score(y_test, ensemble_pred)
ensemble_roc_auc = roc_auc_score(y_test, ensemble_prob)

print("===== XGBoost + CatBoost + HistGradientBoosting Ensemble =====")
print(f"Accuracy : {ensemble_accuracy:.4f}")
print(f"Precision: {ensemble_precision:.4f}")
print(f"Recall   : {ensemble_recall:.4f}")
print(f"F1 Score : {ensemble_f1:.4f}")
print(f"ROC-AUC  : {ensemble_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, ensemble_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, ensemble_pred))

===== XGBoost + CatBoost + HistGradientBoosting Ensemble =====
Accuracy : 0.6966
Precision: 0.6500
Recall   : 0.5388
F1 Score : 0.5892
ROC-AUC  : 0.7604

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.80      0.76     39544
           1       0.65      0.54      0.59     26791

    accuracy                           0.70     66335
   macro avg       0.69      0.67      0.67     66335
weighted avg       0.69      0.70      0.69     66335


Confusion Matrix:
[[31772  7772]
 [12355 14436]]


In [42]:
# ==========================================
# ENSEMBLE vs INDIVIDUAL MODELS
# ==========================================

ensemble_comparison = pd.DataFrame({
    "Model": [
        "XGBoost",
        "CatBoost",
        "HistGradientBoosting",
        "XGB + CatBoost + HistGB Ensemble"
    ],
    "Accuracy": [
        xgb_accuracy,
        cat_accuracy,
        hgb_accuracy,
        ensemble_accuracy
    ],
    "Precision": [
        xgb_precision,
        cat_precision,
        hgb_precision,
        ensemble_precision
    ],
    "Recall": [
        xgb_recall,
        cat_recall,
        hgb_recall,
        ensemble_recall
    ],
    "F1": [
        xgb_f1,
        cat_f1,
        hgb_f1,
        ensemble_f1
    ],
    "ROC-AUC": [

        xgb_roc_auc,
        cat_roc_auc,
        hgb_roc_auc,
        ensemble_roc_auc
    ]

})

display(
    ensemble_comparison
    .sort_values("Accuracy", ascending=False)
    .round(4)
)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
1,CatBoost,0.6975,0.6527,0.5362,0.5888,0.7614
3,XGB + CatBoost + HistGB Ensemble,0.6966,0.6500,0.5388,0.5892,0.7604
2,HistGradientBoosting,0.6962,0.6436,0.5554,0.5962,0.7599
0,XGBoost,0.6918,0.6475,0.5198,0.5767,0.7549


In [43]:
# ==========================================
# CREATE VALIDATION SET FROM TRAINING DATA
# ==========================================

from sklearn.model_selection import train_test_split

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Training subset:", X_train_sub.shape)
print("Validation set :", X_val.shape)
print("Final test set  :", X_test.shape)

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True).round(4))

Training subset: (212269, 219)
Validation set : (53068, 219)
Final test set  : (66335, 219)

Validation target distribution:
target
0    0.5961
1    0.4039
Name: proportion, dtype: float64


In [44]:
# =========================================================
# RETRAIN MODELS USING TRAINING SUBSET ONLY
# =========================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# ---------------------------------------------------------
# 1. XGBoost
# ---------------------------------------------------------

xgb_val_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_val_model.fit(X_train_sub, y_train_sub)

print("XGBoost retrained successfully!")


# ---------------------------------------------------------
# 2. CatBoost
# ---------------------------------------------------------

cat_val_model = CatBoostClassifier(
    iterations=300,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    thread_count=-1
)

cat_val_model.fit(X_train_sub, y_train_sub)

print("CatBoost retrained successfully!")


# ---------------------------------------------------------
# 3. HistGradientBoosting
# ---------------------------------------------------------

hgb_val_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

hgb_val_model.fit(X_train_sub, y_train_sub)

print("HistGradientBoosting retrained successfully!")

XGBoost retrained successfully!
CatBoost retrained successfully!
HistGradientBoosting retrained successfully!


In [45]:
# =========================================================
# SVM + KNN PREPARATION
# =========================================================

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed!")
print("Train shape:", X_train_scaled.shape)
print("Test shape :", X_test_scaled.shape)

Scaling completed!
Train shape: (265337, 219)
Test shape : (66335, 219)


In [46]:
# =========================================================
# FAST SVM + KNN BENCHMARK
# =========================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Use only 20,000 samples
X_small, _, y_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=20000,
    stratify=y_train,
    random_state=42
)

X_svm_train, X_svm_test, y_svm_train, y_svm_test = train_test_split(
    X_small,
    y_small,
    test_size=0.25,
    stratify=y_small,
    random_state=42
)

scaler_small = StandardScaler()

X_svm_train = scaler_small.fit_transform(X_svm_train)
X_svm_test = scaler_small.transform(X_svm_test)

print("Small benchmark dataset ready!")
print("Train:", X_svm_train.shape)
print("Test :", X_svm_test.shape)

Small benchmark dataset ready!
Train: (15000, 219)
Test : (5000, 219)


In [47]:
from sklearn.svm import LinearSVC

svm_fast = LinearSVC(
    C=1.0,
    max_iter=3000,
    random_state=42
)

svm_fast.fit(X_svm_train, y_svm_train)

y_pred_svm = svm_fast.predict(X_svm_test)

print("===== FAST SVM =====")
print(f"Accuracy: {accuracy_score(y_svm_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_svm_test, y_pred_svm):.4f}")
print(f"Recall: {recall_score(y_svm_test, y_pred_svm):.4f}")
print(f"F1: {f1_score(y_svm_test, y_pred_svm):.4f}")

print(classification_report(y_svm_test, y_pred_svm))

===== FAST SVM =====
Accuracy: 0.6670
Precision: 0.6157
Recall: 0.4666
F1: 0.5309
              precision    recall  f1-score   support

           0       0.69      0.80      0.74      2981
           1       0.62      0.47      0.53      2019

    accuracy                           0.67      5000
   macro avg       0.65      0.63      0.64      5000
weighted avg       0.66      0.67      0.66      5000



In [48]:
knn_fast = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    n_jobs=-1
)

knn_fast.fit(X_svm_train, y_svm_train)

y_pred_knn = knn_fast.predict(X_svm_test)

print("===== FAST KNN =====")
print(f"Accuracy: {accuracy_score(y_svm_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_svm_test, y_pred_knn):.4f}")
print(f"Recall: {recall_score(y_svm_test, y_pred_knn):.4f}")
print(f"F1: {f1_score(y_svm_test, y_pred_knn):.4f}")

print(classification_report(y_svm_test, y_pred_knn))

===== FAST KNN =====
Accuracy: 0.6280
Precision: 0.5431
Recall: 0.4963
F1: 0.5186
              precision    recall  f1-score   support

           0       0.68      0.72      0.70      2981
           1       0.54      0.50      0.52      2019

    accuracy                           0.63      5000
   macro avg       0.61      0.61      0.61      5000
weighted avg       0.62      0.63      0.62      5000



In [49]:
# =========================================================
# FINAL MODEL COMPARISON
# =========================================================

final_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "HistGradientBoosting",
        "CatBoost",
        "Tuned CatBoost"
    ],
    "Accuracy": [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy,
        hgb_accuracy,
        cat_accuracy,
        cat_tuned_accuracy
    ],
    "Precision": [
        lr_precision,
        rf_precision,
        xgb_precision,
        hgb_precision,
        cat_precision,
        cat_tuned_precision
    ],
    "Recall": [
        lr_recall,
        rf_recall,
        xgb_recall,
        hgb_recall,
        cat_recall,
        cat_tuned_recall
    ],
    "F1": [
        lr_f1,
        rf_f1,
        xgb_f1,
        hgb_f1,
        cat_f1,
        cat_tuned_f1
    ],
    "ROC-AUC": [
        lr_roc_auc,
        rf_roc_auc,
        xgb_roc_auc,
        hgb_roc_auc,
        cat_roc_auc,
        cat_tuned_roc_auc
    ]
})

final_comparison = final_comparison.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

display(final_comparison.round(4))

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Tuned CatBoost,0.6993,0.6523,0.5471,0.5951,0.7643
1,CatBoost,0.6975,0.6527,0.5362,0.5888,0.7614
2,HistGradientBoosting,0.6962,0.6436,0.5554,0.5962,0.7599
3,XGBoost,0.6918,0.6475,0.5198,0.5767,0.7549
4,Random Forest,0.6658,0.5713,0.6914,0.6257,0.7358
5,Logistic Regression,0.6122,0.5699,0.1625,0.2529,0.6500


In [50]:
# =========================================================
# SAVE FINAL CATBOOST MODEL
# =========================================================

import os
import joblib
import json

MODEL_DIR = "/content/drive/MyDrive/kickstarter-success-prediction (1)/models"

os.makedirs(MODEL_DIR, exist_ok=True)

# Save model
cat_tuned.save_model(
    os.path.join(MODEL_DIR, "kickstarter_catboost_final.cbm")
)

# Save feature names
joblib.dump(
    list(X_train.columns),
    os.path.join(MODEL_DIR, "kickstarter_feature_columns.pkl")
)

# Save model information
model_info = {
    "model": "Tuned CatBoost",
    "accuracy": float(cat_tuned_accuracy),
    "precision": float(cat_tuned_precision),
    "recall": float(cat_tuned_recall),
    "f1_score": float(cat_tuned_f1),
    "roc_auc": float(cat_tuned_roc_auc),
    "number_of_features": int(X_train.shape[1]),
    "features": list(X_train.columns)
}

with open(
    os.path.join(MODEL_DIR, "model_info.json"),
    "w"
) as f:
    json.dump(model_info, f, indent=4)

print("========================================")
print("FINAL MODEL SAVED SUCCESSFULLY")
print("========================================")
print("Model       : Tuned CatBoost")
print("Accuracy    :", round(cat_tuned_accuracy, 4))
print("Precision   :", round(cat_tuned_precision, 4))
print("Recall      :", round(cat_tuned_recall, 4))
print("F1 Score    :", round(cat_tuned_f1, 4))
print("ROC-AUC     :", round(cat_tuned_roc_auc, 4))
print("Features    :", X_train.shape[1])
print("Location    :", MODEL_DIR)

FINAL MODEL SAVED SUCCESSFULLY
Model       : Tuned CatBoost
Accuracy    : 0.6993
Precision   : 0.6523
Recall      : 0.5471
F1 Score    : 0.5951
ROC-AUC     : 0.7643
Features    : 219
Location    : /content/drive/MyDrive/kickstarter-success-prediction (1)/models


In [51]:
# =========================================================
# TEST FINAL SAVED MODEL
# =========================================================

from catboost import CatBoostClassifier
import joblib
import os

MODEL_DIR = "/content/drive/MyDrive/kickstarter-success-prediction (1)/models"

# Load saved model
final_model = CatBoostClassifier()
final_model.load_model(
    os.path.join(MODEL_DIR, "kickstarter_catboost_final.cbm")
)

# Load feature columns
feature_columns = joblib.load(
    os.path.join(MODEL_DIR, "kickstarter_feature_columns.pkl")
)

# Make sure test data has the same feature order
X_test_final = X_test[feature_columns]

# Predict
test_pred = final_model.predict(
    X_test_final
).astype(int).ravel()

test_prob = final_model.predict_proba(
    X_test_final
)[:, 1]

print("=" * 55)
print("FINAL MODEL TEST")
print("=" * 55)

print("Test samples :", len(X_test_final))
print("Features     :", X_test_final.shape[1])

print("\nAccuracy  :", round(
    accuracy_score(y_test, test_pred), 4
))

print("Precision :", round(
    precision_score(y_test, test_pred), 4
))

print("Recall    :", round(
    recall_score(y_test, test_pred), 4
))

print("F1 Score  :", round(
    f1_score(y_test, test_pred), 4
))

print("ROC-AUC   :", round(
    roc_auc_score(y_test, test_prob), 4
))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

FINAL MODEL TEST
Test samples : 66335
Features     : 219

Accuracy  : 0.6993
Precision : 0.6523
Recall    : 0.5471
F1 Score  : 0.5951
ROC-AUC   : 0.7643

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.80      0.76     39544
           1       0.65      0.55      0.60     26791

    accuracy                           0.70     66335
   macro avg       0.69      0.67      0.68     66335
weighted avg       0.69      0.70      0.69     66335



In [53]:
# =========================================================
# DATA QUALITY DIAGNOSTIC - FIXED
# Uses X and y already present in memory
# =========================================================

print("=" * 60)
print("DATA QUALITY + TARGET DIAGNOSTIC")
print("=" * 60)

# ---------------------------------------------------------
# 1. TARGET DISTRIBUTION
# ---------------------------------------------------------
print("\n1. TARGET DISTRIBUTION")

print(y.value_counts())
print("\nProportion:")
print(y.value_counts(normalize=True).round(4))


# ---------------------------------------------------------
# 2. DUPLICATE FEATURE ROWS
# ---------------------------------------------------------
print("\n2. DUPLICATE FEATURE ROWS")

duplicate_rows = X.duplicated().sum()

print("Duplicate feature rows:", duplicate_rows)


# ---------------------------------------------------------
# 3. MISSING VALUES
# ---------------------------------------------------------
print("\n3. MISSING VALUES")

missing_total = X.isna().sum().sum()

print("Total missing values:", missing_total)

missing_features = X.columns[X.isna().any()].tolist()

print("Features with missing values:", len(missing_features))

if missing_features:
    print(missing_features)


# ---------------------------------------------------------
# 4. CONSTANT FEATURES
# ---------------------------------------------------------
print("\n4. CONSTANT FEATURES")

constant_cols = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]

print("Constant features:", len(constant_cols))

if constant_cols:
    print(constant_cols)


# ---------------------------------------------------------
# 5. LOW-CARDINALITY FEATURES
# ---------------------------------------------------------
print("\n5. LOW-CARDINALITY FEATURES")

low_cardinality = []

for col in X.columns:
    n_unique = X[col].nunique(dropna=False)

    if n_unique <= 3:
        low_cardinality.append((col, n_unique))

for col, n_unique in low_cardinality:
    print(f"{col:35s} -> {n_unique} unique values")


# ---------------------------------------------------------
# 6. TRAIN / TEST EXACT OVERLAP
# ---------------------------------------------------------
print("\n6. TRAIN / TEST OVERLAP")

train_hash = pd.util.hash_pandas_object(
    X_train,
    index=False
)

test_hash = pd.util.hash_pandas_object(
    X_test,
    index=False
)

overlap = len(set(train_hash) & set(test_hash))

print("Exact feature-row overlap:", overlap)


# ---------------------------------------------------------
# 7. TARGET CORRELATION
# ---------------------------------------------------------
print("\n7. TOP NUMERICAL FEATURES RELATED TO TARGET")

numeric_cols = X.select_dtypes(include="number").columns

correlations = (
    X[numeric_cols]
    .corrwith(y)
    .abs()
    .sort_values(ascending=False)
)

print(correlations.head(20))


# ---------------------------------------------------------
# FINAL SUMMARY
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("DIAGNOSTIC COMPLETED")
print("=" * 60)

print("Total features       :", X.shape[1])
print("Total samples        :", X.shape[0])
print("Duplicate rows       :", duplicate_rows)
print("Missing values       :", missing_total)
print("Constant features    :", len(constant_cols))
print("Train/Test overlap   :", overlap)

DATA QUALITY + TARGET DIAGNOSTIC

1. TARGET DISTRIBUTION
target
0    197716
1    133956
Name: count, dtype: int64

Proportion:
target
0    0.5961
1    0.4039
Name: proportion, dtype: float64

2. DUPLICATE FEATURE ROWS
Duplicate feature rows: 0

3. MISSING VALUES
Total missing values: 0
Features with missing values: 0

4. CONSTANT FEATURES
Constant features: 0

5. LOW-CARDINALITY FEATURES
category_Academic                   -> 2 unique values
category_Accessories                -> 2 unique values
category_Action                     -> 2 unique values
category_Animals                    -> 2 unique values
category_Animation                  -> 2 unique values
category_Anthologies                -> 2 unique values
category_Apparel                    -> 2 unique values
category_Apps                       -> 2 unique values
category_Architecture               -> 2 unique values
category_Art                        -> 2 unique values
category_Art Books                  -> 2 unique values
cate

In [54]:
# ============================================================
# KICKSTARTER FEATURE ENGINEERING
# ============================================================

import numpy as np
import pandas as pd

X_fe = X.copy()

print("Original features:", X_fe.shape[1])

# ------------------------------------------------------------
# 1. GOAL-BASED FEATURES
# ------------------------------------------------------------

if "goal" in X_fe.columns:
    X_fe["log_goal"] = np.log1p(X_fe["goal"])

if "usd_goal_real" in X_fe.columns:
    X_fe["log_usd_goal"] = np.log1p(X_fe["usd_goal_real"])

# ------------------------------------------------------------
# 2. CAMPAIGN DURATION FEATURES
# ------------------------------------------------------------

if "duration_days" in X_fe.columns:
    X_fe["duration_squared"] = X_fe["duration_days"] ** 2
    X_fe["duration_log"] = np.log1p(X_fe["duration_days"])

    # useful duration ranges
    X_fe["duration_short"] = (
        X_fe["duration_days"] <= 14
    ).astype(int)

    X_fe["duration_medium"] = (
        (X_fe["duration_days"] > 14) &
        (X_fe["duration_days"] <= 30)
    ).astype(int)

    X_fe["duration_long"] = (
        X_fe["duration_days"] > 30
    ).astype(int)

# ------------------------------------------------------------
# 3. NAME FEATURES
# ------------------------------------------------------------

if "name_length" in X_fe.columns:

    X_fe["name_length_squared"] = (
        X_fe["name_length"] ** 2
    )

    X_fe["name_length_log"] = np.log1p(
        X_fe["name_length"]
    )

if "name_word_count" in X_fe.columns:

    X_fe["name_word_count_squared"] = (
        X_fe["name_word_count"] ** 2
    )

    X_fe["name_word_count_log"] = np.log1p(
        X_fe["name_word_count"]
    )

# ------------------------------------------------------------
# 4. DATE FEATURES
# ------------------------------------------------------------

if "launch_year" in X_fe.columns and "deadline_year" in X_fe.columns:

    X_fe["campaign_year_gap"] = (
        X_fe["deadline_year"] -
        X_fe["launch_year"]
    )

# ------------------------------------------------------------
# 5. GOAL / DURATION INTERACTION
# ------------------------------------------------------------

if "usd_goal_real" in X_fe.columns and "duration_days" in X_fe.columns:

    X_fe["goal_per_day"] = (
        X_fe["usd_goal_real"] /
        (X_fe["duration_days"] + 1)
    )

    X_fe["log_goal_per_day"] = np.log1p(
        X_fe["goal_per_day"]
    )

# ------------------------------------------------------------
# 6. GOAL / NAME COMPLEXITY
# ------------------------------------------------------------

if (
    "usd_goal_real" in X_fe.columns and
    "name_word_count" in X_fe.columns
):

    X_fe["goal_per_word"] = (
        X_fe["usd_goal_real"] /
        (X_fe["name_word_count"] + 1)
    )

# ------------------------------------------------------------
# 7. PRINT RESULTS
# ------------------------------------------------------------

print("\nFeature engineering completed!")

print("Original features :", X.shape[1])
print("New features      :", X_fe.shape[1])
print("Added features    :", X_fe.shape[1] - X.shape[1])

print("\nNew feature names:")

new_features = [
    col for col in X_fe.columns
    if col not in X.columns
]

print(new_features)

Original features: 219

Feature engineering completed!
Original features : 219
New features      : 234
Added features    : 15

New feature names:
['log_goal', 'log_usd_goal', 'duration_squared', 'duration_log', 'duration_short', 'duration_medium', 'duration_long', 'name_length_squared', 'name_length_log', 'name_word_count_squared', 'name_word_count_log', 'campaign_year_gap', 'goal_per_day', 'log_goal_per_day', 'goal_per_word']


In [55]:
# ============================================================
# RETRAIN CATBOOST WITH ENGINEERED FEATURES
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# ------------------------------------------------------------
# Use the SAME train/test indices as the original split
# ------------------------------------------------------------

X_fe_train = X_fe.loc[X_train.index]
X_fe_test  = X_fe.loc[X_test.index]

print("Training shape :", X_fe_train.shape)
print("Testing shape  :", X_fe_test.shape)

# ------------------------------------------------------------
# Train CatBoost
# ------------------------------------------------------------

cat_fe_model = CatBoostClassifier(
    iterations=700,
    depth=7,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

cat_fe_model.fit(
    X_fe_train,
    y_train
)

print("\nCatBoost with engineered features trained successfully!")

# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

y_pred_fe = cat_fe_model.predict(
    X_fe_test
).astype(int).ravel()

y_prob_fe = cat_fe_model.predict_proba(
    X_fe_test
)[:, 1]

# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

fe_accuracy = accuracy_score(y_test, y_pred_fe)
fe_precision = precision_score(y_test, y_pred_fe)
fe_recall = recall_score(y_test, y_pred_fe)
fe_f1 = f1_score(y_test, y_pred_fe)
fe_roc_auc = roc_auc_score(y_test, y_prob_fe)

print("\n" + "=" * 55)
print("CATBOOST + FEATURE ENGINEERING")
print("=" * 55)

print(f"Accuracy  : {fe_accuracy:.4f}")
print(f"Precision : {fe_precision:.4f}")
print(f"Recall    : {fe_recall:.4f}")
print(f"F1 Score  : {fe_f1:.4f}")
print(f"ROC-AUC   : {fe_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_fe))

Training shape : (265337, 234)
Testing shape  : (66335, 234)
0:	total: 438ms	remaining: 5m 6s
100:	total: 20.7s	remaining: 2m 2s
200:	total: 30.3s	remaining: 1m 15s
300:	total: 37.7s	remaining: 49.9s
400:	total: 47.1s	remaining: 35.1s
500:	total: 54.5s	remaining: 21.6s
600:	total: 1m 4s	remaining: 10.6s
699:	total: 1m 13s	remaining: 0us

CatBoost with engineered features trained successfully!

CATBOOST + FEATURE ENGINEERING
Accuracy  : 0.6997
Precision : 0.6531
Recall    : 0.5471
F1 Score  : 0.5954
ROC-AUC   : 0.7643

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.80      0.76     39544
           1       0.65      0.55      0.60     26791

    accuracy                           0.70     66335
   macro avg       0.69      0.68      0.68     66335
weighted avg       0.70      0.70      0.69     66335



In [56]:
# ============================================================
# CATBOOST TARGETED TUNING
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

results = []

configs = [
    {
        "depth": 5,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5,
        "class_weights": [1, 1.0]
    },
    {
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5,
        "class_weights": [1, 1.15]
    },
    {
        "depth": 7,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5,
        "class_weights": [1, 1.15]
    },
    {
        "depth": 6,
        "learning_rate": 0.08,
        "l2_leaf_reg": 10,
        "class_weights": [1, 1.20]
    }
]

for i, cfg in enumerate(configs):

    print(f"\n{'='*60}")
    print(f"MODEL {i+1}/{len(configs)}")
    print(cfg)
    print(f"{'='*60}")

    model = CatBoostClassifier(
        iterations=700,
        depth=cfg["depth"],
        learning_rate=cfg["learning_rate"],
        l2_leaf_reg=cfg["l2_leaf_reg"],
        class_weights=cfg["class_weights"],
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=200,
        allow_writing_files=False
    )

    model.fit(
        X_train_sub,
        y_train_sub,
        eval_set=(X_val, y_val),
        early_stopping_rounds=80
    )

    # Validation probabilities
    prob = model.predict_proba(X_val)[:, 1]

    # Default threshold
    pred = (prob >= 0.50).astype(int)

    acc = accuracy_score(y_val, pred)
    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, zero_division=0)
    roc = roc_auc_score(y_val, prob)

    results.append({
        "Model": i + 1,
        "Depth": cfg["depth"],
        "Learning Rate": cfg["learning_rate"],
        "L2": cfg["l2_leaf_reg"],
        "Class Weight": cfg["class_weights"][1],
        "Iterations": model.get_best_iteration(),
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc,
        "Model Object": model
    })

    print("\nValidation Results:")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1        : {f1:.4f}")
    print(f"ROC-AUC   : {roc:.4f}")

results_df = pd.DataFrame(results)

print("\n" + "="*80)
print("CATBOOST TUNING RESULTS")
print("="*80)

display(
    results_df[
        [
            "Model",
            "Depth",
            "Learning Rate",
            "L2",
            "Class Weight",
            "Iterations",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "ROC-AUC"
        ]
    ].sort_values("F1", ascending=False).round(4)
)


MODEL 1/4
{'depth': 5, 'learning_rate': 0.05, 'l2_leaf_reg': 5, 'class_weights': [1, 1.0]}
0:	test: 0.6579592	best: 0.6579592 (0)	total: 153ms	remaining: 1m 46s
200:	test: 0.7439114	best: 0.7439114 (200)	total: 18s	remaining: 44.7s
400:	test: 0.7539222	best: 0.7539222 (400)	total: 30.7s	remaining: 22.9s
600:	test: 0.7587705	best: 0.7587705 (600)	total: 43.5s	remaining: 7.17s
699:	test: 0.7602180	best: 0.7602180 (699)	total: 49s	remaining: 0us

bestTest = 0.7602180376
bestIteration = 699


Validation Results:
Accuracy  : 0.7001
Precision : 0.6565
Recall    : 0.5398
F1        : 0.5925
ROC-AUC   : 0.7602

MODEL 2/4
{'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 5, 'class_weights': [1, 1.15]}
0:	test: 0.6631579	best: 0.6631579 (0)	total: 60.7ms	remaining: 42.5s
200:	test: 0.7473105	best: 0.7473105 (200)	total: 13.1s	remaining: 32.6s
400:	test: 0.7571445	best: 0.7571445 (400)	total: 25.5s	remaining: 19s
600:	test: 0.7615535	best: 0.7615535 (600)	total: 38.9s	remaining: 6.4s
699:	test: 

,Model,Depth,Learning Rate,L2,Class Weight,Iterations,Accuracy,Precision,Recall,F1,ROC-AUC
3,4,6,0.08,10,1.20,699,0.7035,0.6314,0.6387,0.6350,0.7658
2,3,7,0.05,5,1.15,699,0.7028,0.6366,0.6152,0.6257,0.7641
1,2,6,0.05,5,1.15,699,0.7024,0.6364,0.6137,0.6249,0.7628
0,1,5,0.05,5,1.00,699,0.7001,0.6565,0.5398,0.5925,0.7602


In [57]:
# ============================================================
# FIND BEST THRESHOLD FOR BEST CATBOOST MODEL
# ============================================================

best_model = results_df.loc[
    results_df["F1"].idxmax(),
    "Model Object"
]

val_prob = best_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in [i / 100 for i in range(20, 71)]:

    val_pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred, zero_division=0),
        "Recall": recall_score(y_val, val_pred, zero_division=0),
        "F1": f1_score(y_val, val_pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

# Best threshold according to F1
best_threshold_row = threshold_df.loc[
    threshold_df["F1"].idxmax()
]

print("=" * 60)
print("BEST CATBOOST THRESHOLD")
print("=" * 60)

display(best_threshold_row.to_frame().T.round(4))

print("\nTop 10 thresholds by F1:")
display(
    threshold_df
    .sort_values("F1", ascending=False)
    .head(10)
    .round(4)
)

BEST CATBOOST THRESHOLD


,Threshold,Accuracy,Precision,Recall,F1
18,0.38,0.6627,0.5564,0.8134,0.6608



Top 10 thresholds by F1:


,Threshold,Accuracy,Precision,Recall,F1
18,0.38,0.6627,0.5564,0.8134,0.6608
17,0.37,0.6571,0.5503,0.8248,0.6602
19,0.39,0.6671,0.5618,0.7992,0.6598
20,0.40,0.6724,0.5682,0.7861,0.6596
16,0.36,0.6512,0.5444,0.8361,0.6594
21,0.41,0.6768,0.5742,0.7728,0.6589
15,0.35,0.6449,0.5384,0.8467,0.6582
14,0.34,0.6399,0.5337,0.8583,0.6581
22,0.42,0.6811,0.5804,0.7592,0.6579
23,0.43,0.6851,0.5867,0.7458,0.6568


In [58]:
# ============================================================
# FINAL CATBOOST MODEL
# Train on ALL training data
# ============================================================

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

FINAL_THRESHOLD = 0.38

final_catboost = CatBoostClassifier(
    iterations=700,
    depth=6,
    learning_rate=0.08,
    l2_leaf_reg=10,
    class_weights=[1, 1.2],
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False
)

print("Training FINAL CatBoost...")
print("Training data:", X_train.shape)

final_catboost.fit(
    X_train,
    y_train
)

print("\nFinal CatBoost trained successfully!")

# ============================================================
# FINAL TEST PREDICTIONS
# ============================================================

test_prob = final_catboost.predict_proba(X_test)[:, 1]

# Use optimized threshold
test_pred = (test_prob >= FINAL_THRESHOLD).astype(int)

# ============================================================
# FINAL TEST METRICS
# ============================================================

final_accuracy = accuracy_score(y_test, test_pred)
final_precision = precision_score(
    y_test, test_pred, zero_division=0
)
final_recall = recall_score(
    y_test, test_pred, zero_division=0
)
final_f1 = f1_score(
    y_test, test_pred, zero_division=0
)
final_roc_auc = roc_auc_score(
    y_test, test_prob
)

print("\n" + "=" * 60)
print("FINAL CATBOOST TEST RESULTS")
print("=" * 60)

print(f"Threshold : {FINAL_THRESHOLD}")
print(f"Accuracy  : {final_accuracy:.4f}")
print(f"Precision : {final_precision:.4f}")
print(f"Recall    : {final_recall:.4f}")
print(f"F1 Score  : {final_f1:.4f}")
print(f"ROC-AUC   : {final_roc_auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

Training FINAL CatBoost...
Training data: (265337, 219)
0:	total: 127ms	remaining: 1m 28s
200:	total: 17.7s	remaining: 43.9s
400:	total: 32.2s	remaining: 24s
600:	total: 46.9s	remaining: 7.73s
699:	total: 55.3s	remaining: 0us

Final CatBoost trained successfully!

FINAL CATBOOST TEST RESULTS
Threshold : 0.38
Accuracy  : 0.6629
Precision : 0.5563
Recall    : 0.8169
F1 Score  : 0.6618
ROC-AUC   : 0.7653

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.56      0.66     39544
           1       0.56      0.82      0.66     26791

    accuracy                           0.66     66335
   macro avg       0.69      0.69      0.66     66335
weighted avg       0.71      0.66      0.66     66335


Confusion Matrix:
[[22086 17458]
 [ 4906 21885]]


In [59]:
# ============================================================
# SAVE FINAL CATBOOST MODEL + METADATA
# ============================================================

import os
import json
import joblib

# ------------------------------------------------------------
# MODEL DIRECTORY
# ------------------------------------------------------------

MODEL_DIR = "/content/drive/MyDrive/kickstarter-sucess-prediction (1)/models"

os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. SAVE CATBOOST MODEL
# ------------------------------------------------------------

FINAL_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "kickstarter_catboost_final.cbm"
)

final_catboost.save_model(FINAL_MODEL_PATH)

print("CatBoost model saved:")
print(FINAL_MODEL_PATH)


# ------------------------------------------------------------
# 2. SAVE FEATURE NAMES
# ------------------------------------------------------------

FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "feature_names.json"
)

feature_names = list(X_train.columns)

with open(FEATURE_PATH, "w") as f:
    json.dump(feature_names, f, indent=4)

print("\nFeature names saved:")
print(FEATURE_PATH)


# ------------------------------------------------------------
# 3. SAVE DECISION THRESHOLD
# ------------------------------------------------------------

THRESHOLD_PATH = os.path.join(
    MODEL_DIR,
    "threshold.json"
)

threshold_info = {
    "threshold": FINAL_THRESHOLD,
    "selection_metric": "F1",
    "validation_best_f1": 0.6608
}

with open(THRESHOLD_PATH, "w") as f:
    json.dump(threshold_info, f, indent=4)

print("\nThreshold saved:")
print(THRESHOLD_PATH)


# ------------------------------------------------------------
# 4. SAVE MODEL METADATA
# ------------------------------------------------------------

METADATA_PATH = os.path.join(
    MODEL_DIR,
    "model_metadata.json"
)

model_metadata = {
    "model": "CatBoostClassifier",

    "parameters": {
        "iterations": 700,
        "depth": 6,
        "learning_rate": 0.08,
        "l2_leaf_reg": 10,
        "class_weights": [1, 1.2],
        "random_seed": 42
    },

    "features": len(feature_names),

    "decision_threshold": FINAL_THRESHOLD,

    "validation_metrics": {
        "accuracy": 0.7035,
        "precision": 0.6314,
        "recall": 0.6387,
        "f1": 0.6350,
        "roc_auc": 0.7658
    },

    "test_metrics": {
        "accuracy": final_accuracy,
        "precision": final_precision,
        "recall": final_recall,
        "f1": final_f1,
        "roc_auc": final_roc_auc
    },

    "test_samples": len(X_test)
}

with open(METADATA_PATH, "w") as f:
    json.dump(model_metadata, f, indent=4)

print("\nMetadata saved:")
print(METADATA_PATH)


# ------------------------------------------------------------
# 5. VERIFY FILES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL MODEL FILES")
print("=" * 60)

for file in os.listdir(MODEL_DIR):
    print(file)

print("\nFINAL MODEL SAVED SUCCESSFULLY!")

CatBoost model saved:
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/models/kickstarter_catboost_final.cbm

Feature names saved:
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/models/feature_names.json

Threshold saved:
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/models/threshold.json

Metadata saved:
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/models/model_metadata.json

FINAL MODEL FILES
kickstarter_catboost_final.cbm
feature_names.json
threshold.json
model_metadata.json

FINAL MODEL SAVED SUCCESSFULLY!


In [60]:
# ============================================================
# CHECK FINAL PREDICTION PIPELINE VARIABLES
# ============================================================

print("Checking required objects...\n")

required = [
    "final_catboost",
    "X_train",
    "feature_names",
    "FINAL_THRESHOLD"
]

for name in required:
    if name in globals():
        obj = globals()[name]
        print(f"✓ {name:<25} {type(obj).__name__}")
    else:
        print(f"✗ {name:<25} NOT FOUND")

print("\nFeature count:", len(X_train.columns))
print("Threshold:", FINAL_THRESHOLD)

Checking required objects...

✓ final_catboost            CatBoostClassifier
✓ X_train                   DataFrame
✓ feature_names             list
✓ FINAL_THRESHOLD           float

Feature count: 219
Threshold: 0.38


In [62]:
# ============================================================
# VERIFY FINAL SAVED MODEL
# ============================================================

import os
import json
from catboost import CatBoostClassifier

MODEL_DIR = "/content/drive/MyDrive/kickstarter-sucess-prediction (1)/models"

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "kickstarter_catboost_final.cbm"
)

FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "feature_names.json"
)

THRESHOLD_PATH = os.path.join(
    MODEL_DIR,
    "threshold.json"
)

# Load model
loaded_model = CatBoostClassifier()
loaded_model.load_model(MODEL_PATH)

# Load feature names
with open(FEATURE_PATH, "r") as f:
    saved_features = json.load(f)

# Load threshold
with open(THRESHOLD_PATH, "r") as f:
    threshold_data = json.load(f)

print("=" * 60)
print("FINAL MODEL VERIFICATION")
print("=" * 60)

print("Model loaded       :", type(loaded_model).__name__)
print("Saved feature list :", len(saved_features))
print("Threshold          :", threshold_data)

print("\nFirst 10 features:")
print(saved_features[:10])

print("\nLast 10 features:")
print(saved_features[-10:])

print("=" * 60)

FINAL MODEL VERIFICATION
Model loaded       : CatBoostClassifier
Saved feature list : 219
Threshold          : {'threshold': 0.38, 'selection_metric': 'F1', 'validation_best_f1': 0.6608}

First 10 features:
['goal', 'usd_goal_real', 'duration_days', 'launch_year', 'launch_month', 'launch_day', 'launch_weekday', 'deadline_year', 'deadline_month', 'deadline_weekday']

Last 10 features:
['country_JP', 'country_LU', 'country_MX', 'country_N,0"', 'country_NL', 'country_NO', 'country_NZ', 'country_SE', 'country_SG', 'country_US']


In [63]:
# ============================================================
# TEST SAVED CATBOOST MODEL
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Make sure test columns match saved feature order
X_test_saved = X_test[saved_features].copy()

# Predict probabilities
test_prob_saved = loaded_model.predict_proba(X_test_saved)[:, 1]

# Apply saved threshold
FINAL_THRESHOLD = threshold_data["threshold"]

test_pred_saved = (
    test_prob_saved >= FINAL_THRESHOLD
).astype(int)

# Metrics
saved_accuracy = accuracy_score(y_test, test_pred_saved)
saved_precision = precision_score(y_test, test_pred_saved)
saved_recall = recall_score(y_test, test_pred_saved)
saved_f1 = f1_score(y_test, test_pred_saved)
saved_roc_auc = roc_auc_score(y_test, test_prob_saved)

print("=" * 60)
print("SAVED MODEL TEST")
print("=" * 60)

print(f"Threshold : {FINAL_THRESHOLD:.2f}")
print(f"Accuracy  : {saved_accuracy:.4f}")
print(f"Precision : {saved_precision:.4f}")
print(f"Recall    : {saved_recall:.4f}")
print(f"F1 Score  : {saved_f1:.4f}")
print(f"ROC-AUC   : {saved_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_pred_saved,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred_saved))

SAVED MODEL TEST
Threshold : 0.38
Accuracy  : 0.6629
Precision : 0.5563
Recall    : 0.8169
F1 Score  : 0.6618
ROC-AUC   : 0.7653

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.56      0.66     39544
           1       0.56      0.82      0.66     26791

    accuracy                           0.66     66335
   macro avg       0.69      0.69      0.66     66335
weighted avg       0.71      0.66      0.66     66335

Confusion Matrix:
[[22086 17458]
 [ 4906 21885]]


In [64]:
# ============================================================
# CHECK ORIGINAL DATA COLUMNS
# ============================================================

print("Dataset shape:", X.shape)

print("\nOriginal feature columns:")
for i, col in enumerate(X.columns):
    print(i, ":", col)

Dataset shape: (331672, 219)

Original feature columns:
0 : goal
1 : usd_goal_real
2 : duration_days
3 : launch_year
4 : launch_month
5 : launch_day
6 : launch_weekday
7 : deadline_year
8 : deadline_month
9 : deadline_weekday
10 : name_length
11 : name_word_count
12 : category_Academic
13 : category_Accessories
14 : category_Action
15 : category_Animals
16 : category_Animation
17 : category_Anthologies
18 : category_Apparel
19 : category_Apps
20 : category_Architecture
21 : category_Art
22 : category_Art Books
23 : category_Audio
24 : category_Bacon
25 : category_Blues
26 : category_Calendars
27 : category_Camera Equipment
28 : category_Candles
29 : category_Ceramics
30 : category_Children's Books
31 : category_Childrenswear
32 : category_Chiptune
33 : category_Civic Design
34 : category_Classical Music
35 : category_Comedy
36 : category_Comic Books
37 : category_Comics
38 : category_Community Gardens
39 : category_Conceptual Art
40 : category_Cookbooks
41 : category_Country & Folk
42 

In [70]:
# ============================================================
# FINAL PREDICTION FUNCTION
# Uses the saved 219-feature CatBoost model
# ============================================================

def predict_new_project(
    name,
    goal,
    usd_goal_real,
    launch_date,
    deadline_date,
    category,
    main_category,
    currency,
    country
):
    import pandas as pd
    import numpy as np

    # --------------------------------------------------------
    # 1. BASIC VALIDATION
    # --------------------------------------------------------

    if not isinstance(name, str) or not name.strip():
        raise ValueError("Project name cannot be empty.")

    if goal <= 0:
        raise ValueError("Goal must be greater than 0.")

    if usd_goal_real <= 0:
        raise ValueError("USD goal must be greater than 0.")

    # --------------------------------------------------------
    # 2. USE SAVED MODEL + FEATURE LIST
    # --------------------------------------------------------

    model = loaded_model

    feature_list = saved_features

    threshold = float(threshold_data["threshold"])

    # --------------------------------------------------------
    # 3. VALID CATEGORIES
    # --------------------------------------------------------

    valid_main_categories = [
        "Comics",
        "Crafts",
        "Dance",
        "Design",
        "Fashion",
        "Film & Video",
        "Games",
        "Journalism",
        "Music",
        "Photography",
        "Publishing",
        "Technology",
        "Theater"
    ]

    if main_category not in valid_main_categories:
        raise ValueError(
            f"Unknown main category: '{main_category}'.\n"
            f"Valid values are: {valid_main_categories}"
        )

    # --------------------------------------------------------
    # 4. VALIDATE CATEGORY AGAINST TRAINED FEATURES
    # --------------------------------------------------------

    category_column = f"category_{category}"

    if category_column not in feature_list:
        raise ValueError(
            f"Unknown category: '{category}'.\n"
            f"'{category_column}' was not present in the trained features."
        )

    # --------------------------------------------------------
    # 5. DATE PROCESSING
    # --------------------------------------------------------

    launch_date = pd.to_datetime(launch_date)
    deadline_date = pd.to_datetime(deadline_date)

    if deadline_date <= launch_date:
        raise ValueError(
            "Deadline date must be after launch date."
        )

    duration_days = (deadline_date - launch_date).days

    # --------------------------------------------------------
    # 6. CREATE EMPTY FEATURE ROW
    # --------------------------------------------------------

    model_input = pd.DataFrame(
        0,
        index=[0],
        columns=feature_list
    )

    # Make sure numeric columns are float-compatible
    model_input = model_input.astype(float)

    # --------------------------------------------------------
    # 7. NUMERICAL FEATURES
    # --------------------------------------------------------

    if "goal" in model_input.columns:
        model_input.loc[0, "goal"] = goal

    if "usd_goal_real" in model_input.columns:
        model_input.loc[0, "usd_goal_real"] = usd_goal_real

    if "duration_days" in model_input.columns:
        model_input.loc[0, "duration_days"] = duration_days

    # --------------------------------------------------------
    # 8. LAUNCH DATE FEATURES
    # --------------------------------------------------------

    if "launch_year" in model_input.columns:
        model_input.loc[0, "launch_year"] = launch_date.year

    if "launch_month" in model_input.columns:
        model_input.loc[0, "launch_month"] = launch_date.month

    if "launch_day" in model_input.columns:
        model_input.loc[0, "launch_day"] = launch_date.day

    if "launch_weekday" in model_input.columns:
        model_input.loc[0, "launch_weekday"] = launch_date.weekday()

    # --------------------------------------------------------
    # 9. DEADLINE DATE FEATURES
    # --------------------------------------------------------

    if "deadline_year" in model_input.columns:
        model_input.loc[0, "deadline_year"] = deadline_date.year

    if "deadline_month" in model_input.columns:
        model_input.loc[0, "deadline_month"] = deadline_date.month

    if "deadline_weekday" in model_input.columns:
        model_input.loc[0, "deadline_weekday"] = deadline_date.weekday()

    # --------------------------------------------------------
    # 10. PROJECT NAME FEATURES
    # --------------------------------------------------------

    name_clean = name.strip()

    name_length = len(name_clean)

    name_word_count = len(
        name_clean.split()
    )

    if "name_length" in model_input.columns:
        model_input.loc[0, "name_length"] = name_length

    if "name_word_count" in model_input.columns:
        model_input.loc[0, "name_word_count"] = name_word_count

    # --------------------------------------------------------
    # 11. CATEGORY ONE-HOT
    # --------------------------------------------------------

    model_input.loc[0, category_column] = 1

    # --------------------------------------------------------
    # 12. MAIN CATEGORY ONE-HOT
    # --------------------------------------------------------

    main_category_column = f"main_category_{main_category}"

    if main_category_column not in feature_list:
        raise ValueError(
            f"Main category column '{main_category_column}' "
            f"is not present in the trained model."
        )

    model_input.loc[0, main_category_column] = 1

    # --------------------------------------------------------
    # 13. CURRENCY ONE-HOT
    # --------------------------------------------------------

    currency_column = f"currency_{currency}"

    if currency_column in feature_list:
        model_input.loc[0, currency_column] = 1
    else:
        raise ValueError(
            f"Unknown currency '{currency}'. "
            f"Expected one of the currencies present in the dataset."
        )

    # --------------------------------------------------------
    # 14. COUNTRY ONE-HOT
    # --------------------------------------------------------

    country_column = f"country_{country}"

    if country_column in feature_list:
        model_input.loc[0, country_column] = 1
    else:
        raise ValueError(
            f"Unknown country '{country}'. "
            f"Expected one of the countries present in the dataset."
        )

    # --------------------------------------------------------
    # 15. FORCE EXACT FEATURE ORDER
    # --------------------------------------------------------

    model_input = model_input[feature_list]

    # --------------------------------------------------------
    # 16. PREDICT PROBABILITY
    # --------------------------------------------------------

    probability = float(
        model.predict_proba(model_input)[0][1]
    )

    # --------------------------------------------------------
    # 17. APPLY OPTIMIZED THRESHOLD
    # --------------------------------------------------------
    # IMPORTANT:
    # probability >= threshold --> SUCCESSFUL
    # probability < threshold  --> NOT SUCCESSFUL

    prediction = int(
        probability >= threshold
    )

    # --------------------------------------------------------
    # 18. DISPLAY RESULT
    # --------------------------------------------------------

    print("=" * 60)
    print("KICKSTARTER SUCCESS PREDICTION")
    print("=" * 60)

    print(f"Project Name     : {name}")
    print(f"Category         : {category}")
    print(f"Main Category    : {main_category}")
    print(f"Country          : {country}")
    print(f"Currency         : {currency}")

    print("-" * 60)

    print(f"Goal             : {goal}")
    print(f"USD Goal         : {usd_goal_real}")
    print(f"Duration         : {duration_days} days")

    print("-" * 60)

    print(
        f"Success Probability : {probability:.2%}"
    )

    print(
        f"Decision Threshold  : {threshold:.2f}"
    )

    print(
        f"Prediction          : "
        f"{'SUCCESSFUL' if prediction == 1 else 'NOT SUCCESSFUL'}"
    )

    print("=" * 60)

    # --------------------------------------------------------
    # 19. RETURN RESULTS
    # --------------------------------------------------------

    return {
        "prediction": prediction,
        "probability": probability,
        "threshold": threshold,
        "features": model_input
    }

In [71]:
result = predict_new_project(
    name="Innovative Smart Home Device",
    goal=10000,
    usd_goal_real=10000,
    launch_date="2017-06-01",
    deadline_date="2017-07-01",
    category="Technology",
    main_category="Technology",
    currency="USD",
    country="US"
)

KICKSTARTER SUCCESS PREDICTION
Project Name     : Innovative Smart Home Device
Category         : Technology
Main Category    : Technology
Country          : US
Currency         : USD
------------------------------------------------------------
Goal             : 10000
USD Goal         : 10000
Duration         : 30 days
------------------------------------------------------------
Success Probability : 24.40%
Decision Threshold  : 0.38
Prediction          : NOT SUCCESSFUL


In [72]:
result = predict_new_project(
    name="Innovative Art and Design Project",
    goal=5000,
    usd_goal_real=5000,
    launch_date="2017-04-01",
    deadline_date="2017-05-31",
    category="Art",
    main_category="Design",
    currency="USD",
    country="US"
)

KICKSTARTER SUCCESS PREDICTION
Project Name     : Innovative Art and Design Project
Category         : Art
Main Category    : Design
Country          : US
Currency         : USD
------------------------------------------------------------
Goal             : 5000
USD Goal         : 5000
Duration         : 60 days
------------------------------------------------------------
Success Probability : 31.01%
Decision Threshold  : 0.38
Prediction          : NOT SUCCESSFUL


In [73]:
result = predict_new_project(
    name="Creative Design Innovation",
    goal=3000,
    usd_goal_real=3000,
    launch_date="2017-04-01",
    deadline_date="2017-06-01",
    category="Graphic Design",
    main_category="Design",
    currency="USD",
    country="US"
)

KICKSTARTER SUCCESS PREDICTION
Project Name     : Creative Design Innovation
Category         : Graphic Design
Main Category    : Design
Country          : US
Currency         : USD
------------------------------------------------------------
Goal             : 3000
USD Goal         : 3000
Duration         : 61 days
------------------------------------------------------------
Success Probability : 23.88%
Decision Threshold  : 0.38
Prediction          : NOT SUCCESSFUL


In [74]:
result = predict_new_project(
    name="Innovative Art and Design Project",
    goal=5000,
    usd_goal_real=5000,
    launch_date="2017-04-01",
    deadline_date="2017-05-31",
    category="Art",
    main_category="Design",
    currency="USD",
    country="US"
)

KICKSTARTER SUCCESS PREDICTION
Project Name     : Innovative Art and Design Project
Category         : Art
Main Category    : Design
Country          : US
Currency         : USD
------------------------------------------------------------
Goal             : 5000
USD Goal         : 5000
Duration         : 60 days
------------------------------------------------------------
Success Probability : 31.01%
Decision Threshold  : 0.38
Prediction          : NOT SUCCESSFUL


In [75]:
print("Available variables containing dataframe-like objects:")

for name, obj in globals().items():
    if hasattr(obj, "shape") and hasattr(obj, "columns"):
        print(name, obj.shape)

Available variables containing dataframe-like objects:
X (331672, 219)
X_train (265337, 219)
X_test (66335, 219)
model_comparison (4, 6)
final_comparison (6, 6)
feature_importance (219, 2)
threshold_df (51, 5)
comparison (6, 6)
X_train_top (265337, 50)
X_test_top (66335, 50)
ensemble_comparison (4, 6)
X_train_sub (212269, 219)
X_val (53068, 219)
X_small (20000, 219)
X_test_final (66335, 219)
X_fe (331672, 234)
X_fe_train (265337, 234)
X_fe_test (66335, 234)
results_df (4, 12)
X_test_saved (66335, 219)


In [76]:
# ============================================================
# REAL TEST-SET VERIFICATION
# Test the saved CatBoost model directly on X_test_saved
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Make sure feature order is exactly the saved feature order
X_check = X_test_saved[saved_features]

# Real test labels
y_check = y_test.reset_index(drop=True)

# Predict probability of SUCCESS (class 1)
test_prob = loaded_model.predict_proba(X_check)[:, 1]

# Apply our optimized threshold
test_pred = (test_prob >= float(threshold_data["threshold"])).astype(int)

# Metrics
accuracy = accuracy_score(y_check, test_pred)
precision = precision_score(y_check, test_pred, zero_division=0)
recall = recall_score(y_check, test_pred, zero_division=0)
f1 = f1_score(y_check, test_pred, zero_division=0)
roc_auc = roc_auc_score(y_check, test_prob)

print("=" * 60)
print("DIRECT SAVED MODEL TEST")
print("=" * 60)

print(f"Test samples       : {len(X_check)}")
print(f"Features           : {X_check.shape[1]}")
print(f"Threshold          : {threshold_data['threshold']:.2f}")

print("-" * 60)

print(f"Accuracy           : {accuracy:.4f}")
print(f"Precision          : {precision:.4f}")
print(f"Recall             : {recall:.4f}")
print(f"F1 Score           : {f1:.4f}")
print(f"ROC-AUC            : {roc_auc:.4f}")

print("=" * 60)

DIRECT SAVED MODEL TEST
Test samples       : 66335
Features           : 219
Threshold          : 0.38
------------------------------------------------------------
Accuracy           : 0.6629
Precision          : 0.5563
Recall             : 0.8169
F1 Score           : 0.6618
ROC-AUC            : 0.7653


In [77]:
# ============================================================
# FIND REAL SUCCESSFUL PROJECTS FROM TEST SET
# ============================================================

successful_indices = np.where(y_check.values == 1)[0]

print("Number of actual successful test projects:", len(successful_indices))

# Show 10 successful projects and what the model predicts
successful_examples = pd.DataFrame({
    "actual": y_check.iloc[successful_indices[:10]].values,
    "success_probability": test_prob[successful_indices[:10]],
    "prediction": test_pred[successful_indices[:10]]
})

successful_examples["success_probability"] = (
    successful_examples["success_probability"] * 100
).round(2)

print(successful_examples)

Number of actual successful test projects: 26791
   actual  success_probability  prediction
0       1                55.00           1
1       1                86.39           1
2       1                60.62           1
3       1                60.42           1
4       1                45.23           1
5       1                74.93           1
6       1                50.57           1
7       1                59.25           1
8       1                27.97           0
9       1                48.72           1


In [78]:
# ============================================================
# BEST REAL SUCCESSFUL PROJECT ACCORDING TO MODEL
# ============================================================

successful_test_probs = test_prob.copy()

# Only consider projects that are actually successful
successful_test_probs[y_check.values != 1] = -1

best_idx = np.argmax(successful_test_probs)

print("=" * 60)
print("BEST REAL SUCCESSFUL TEST EXAMPLE")
print("=" * 60)

print("Test index         :", best_idx)
print("Actual target      :", y_check.iloc[best_idx])
print("Success probability:", f"{test_prob[best_idx]:.2%}")
print("Threshold           :", f"{threshold_data['threshold']:.2f}")
print("Prediction          :", "SUCCESSFUL" if test_pred[best_idx] == 1 else "NOT SUCCESSFUL")
print("=" * 60)

BEST REAL SUCCESSFUL TEST EXAMPLE
Test index         : 39675
Actual target      : 1
Success probability: 98.86%
Threshold           : 0.38
Prediction          : SUCCESSFUL


In [79]:
# ============================================================
# FIND ORIGINAL DATASET
# ============================================================

import os

DATA_DIR = "/content/drive/MyDrive/kickstarter-sucess-prediction (1)"

print("Files in project folder:")
print("=" * 60)

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith((".csv", ".xlsx", ".xls", ".parquet")):
            print(os.path.join(root, file))

Files in project folder:
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/results/final_model_results.csv
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/data/processed/kickstarter_cleaned.csv
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/data/processed/X_encoded.csv
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/data/processed/y.csv
/content/drive/MyDrive/kickstarter-sucess-prediction (1)/data/raw/kickstarter_raw.csv


In [80]:
# ============================================================
# INSPECT FINAL FEATURE DATASET
# ============================================================

import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/kickstarter-sucess-prediction (1)"

FEATURED_PATH = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "X_encoded.csv"
)

RAW_PATH = os.path.join(
    BASE_DIR,
    "data",
    "raw",
    "kickstarter_raw.csv"
)

CLEAN_PATH = os.path.join(
    BASE_DIR,
    "data",
    "processed",
    "kickstarter_cleaned.csv"
)

X_encoded = pd.read_csv(FEATURED_PATH)
raw_df = pd.read_csv(RAW_PATH)
clean_df = pd.read_csv(CLEAN_PATH)

print("=" * 70)
print("DATASET INSPECTION")
print("=" * 70)

print("\nRAW DATA")
print("Shape:", raw_df.shape)
print("Columns:")
print(raw_df.columns.tolist())

print("\n" + "=" * 70)

print("\nCLEANED DATA")
print("Shape:", clean_df.shape)
print("Columns:")
print(clean_df.columns.tolist())

print("\n" + "=" * 70)

print("\nFINAL ENCODED FEATURES")
print("Shape:", X_encoded.shape)
print("Number of features:", len(X_encoded.columns))

print("\nFirst 20 feature columns:")
for i, col in enumerate(X_encoded.columns[:20]):
    print(i, ":", col)

print("\nLast 20 feature columns:")
start = max(0, len(X_encoded.columns) - 20)

for i, col in enumerate(X_encoded.columns[start:], start=start):
    print(i, ":", col)

print("\n" + "=" * 70)

DATASET INSPECTION

RAW DATA
Shape: (378661, 15)
Columns:
['ID', 'name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'pledged', 'state', 'backers', 'country', 'usd pledged', 'usd_pledged_real', 'usd_goal_real']


CLEANED DATA
Shape: (331672, 11)
Columns:
['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'country', 'usd_goal_real', 'target', 'duration_days']


FINAL ENCODED FEATURES
Shape: (331672, 219)
Number of features: 219

First 20 feature columns:
0 : goal
1 : usd_goal_real
2 : duration_days
3 : launch_year
4 : launch_month
5 : launch_day
6 : launch_weekday
7 : deadline_year
8 : deadline_month
9 : deadline_weekday
10 : name_length
11 : name_word_count
12 : category_Academic
13 : category_Accessories
14 : category_Action
15 : category_Animals
16 : category_Animation
17 : category_Anthologies
18 : category_Apparel
19 : category_Apps

Last 20 feature columns:
199 : country_CA
200 : country_CH
201 : country_DE
202 : country_

In [81]:
# ============================================================
# FINAL KICKSTARTER PREDICTION FUNCTION
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE_DIR = "/content/drive/MyDrive/kickstarter-sucess-prediction (1)"
MODEL_DIR = os.path.join(BASE_DIR, "models")

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "kickstarter_catboost_final.cbm"
)

FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "feature_names.json"
)

THRESHOLD_PATH = os.path.join(
    MODEL_DIR,
    "threshold.json"
)


# ------------------------------------------------------------
# LOAD FINAL MODEL
# ------------------------------------------------------------

final_catboost = CatBoostClassifier()
final_catboost.load_model(MODEL_PATH)


# ------------------------------------------------------------
# LOAD EXACT FEATURE LIST
# ------------------------------------------------------------

with open(FEATURE_PATH, "r") as f:
    FEATURE_NAMES = json.load(f)


# ------------------------------------------------------------
# LOAD SAVED THRESHOLD
# ------------------------------------------------------------

with open(THRESHOLD_PATH, "r") as f:
    threshold_data = json.load(f)

FINAL_THRESHOLD = float(threshold_data["threshold"])


print("=" * 70)
print("FINAL PREDICTION PIPELINE READY")
print("=" * 70)

print("Model features :", len(FEATURE_NAMES))
print("Threshold      :", FINAL_THRESHOLD)
print("Model          :", type(final_catboost).__name__)

print("=" * 70)


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_new_project(
    name,
    goal,
    usd_goal_real,
    launch_date,
    deadline_date,
    category,
    main_category,
    currency,
    country
):

    # --------------------------------------------------------
    # CLEAN INPUTS
    # --------------------------------------------------------

    name = str(name).strip()
    category = str(category).strip()
    main_category = str(main_category).strip()
    currency = str(currency).strip().upper()
    country = str(country).strip().upper()

    goal = float(goal)
    usd_goal_real = float(usd_goal_real)

    launch_date = pd.to_datetime(launch_date)
    deadline_date = pd.to_datetime(deadline_date)


    # --------------------------------------------------------
    # BASIC VALIDATION
    # --------------------------------------------------------

    if not name:
        raise ValueError("Project name cannot be empty.")

    if goal <= 0:
        raise ValueError("Goal must be greater than 0.")

    if usd_goal_real <= 0:
        raise ValueError("USD goal must be greater than 0.")

    if deadline_date <= launch_date:
        raise ValueError(
            "Deadline must be after launch date."
        )


    # --------------------------------------------------------
    # DATE FEATURES
    # --------------------------------------------------------

    duration_days = (
        deadline_date - launch_date
    ).days

    launch_year = launch_date.year
    launch_month = launch_date.month
    launch_day = launch_date.day
    launch_weekday = launch_date.weekday()

    deadline_year = deadline_date.year
    deadline_month = deadline_date.month
    deadline_weekday = deadline_date.weekday()


    # --------------------------------------------------------
    # NAME FEATURES
    # --------------------------------------------------------

    name_length = len(name)

    name_word_count = len(
        name.split()
    )


    # --------------------------------------------------------
    # CREATE EXACT 219-FEATURE ROW
    # --------------------------------------------------------

    model_input = pd.DataFrame(
        0.0,
        index=[0],
        columns=FEATURE_NAMES
    )


    # --------------------------------------------------------
    # NUMERICAL FEATURES
    # --------------------------------------------------------

    numerical_features = {

        "goal": goal,

        "usd_goal_real": usd_goal_real,

        "duration_days": duration_days,

        "launch_year": launch_year,

        "launch_month": launch_month,

        "launch_day": launch_day,

        "launch_weekday": launch_weekday,

        "deadline_year": deadline_year,

        "deadline_month": deadline_month,

        "deadline_weekday": deadline_weekday,

        "name_length": name_length,

        "name_word_count": name_word_count
    }


    for feature, value in numerical_features.items():

        if feature in model_input.columns:

            model_input.loc[0, feature] = value


    # --------------------------------------------------------
    # ONE-HOT CATEGORY
    # --------------------------------------------------------

    category_column = f"category_{category}"

    if category_column not in model_input.columns:

        available_categories = [
            col.replace("category_", "")
            for col in model_input.columns
            if col.startswith("category_")
        ]

        raise ValueError(
            f"Unknown category: '{category}'.\n\n"
            f"Use one of these categories:\n"
            f"{available_categories}"
        )

    model_input.loc[0, category_column] = 1


    # --------------------------------------------------------
    # ONE-HOT MAIN CATEGORY
    # --------------------------------------------------------

    main_category_column = (
        f"main_category_{main_category}"
    )

    if main_category_column not in model_input.columns:

        available_main_categories = [
            col.replace("main_category_", "")
            for col in model_input.columns
            if col.startswith("main_category_")
        ]

        raise ValueError(
            f"Unknown main category: "
            f"'{main_category}'.\n\n"
            f"Use one of these main categories:\n"
            f"{available_main_categories}"
        )

    model_input.loc[0, main_category_column] = 1


    # --------------------------------------------------------
    # ONE-HOT CURRENCY
    # --------------------------------------------------------

    currency_column = f"currency_{currency}"

    if currency_column not in model_input.columns:

        available_currencies = [
            col.replace("currency_", "")
            for col in model_input.columns
            if col.startswith("currency_")
        ]

        raise ValueError(
            f"Unknown currency: '{currency}'.\n\n"
            f"Available currencies:\n"
            f"{available_currencies}"
        )

    model_input.loc[0, currency_column] = 1


    # --------------------------------------------------------
    # ONE-HOT COUNTRY
    # --------------------------------------------------------

    country_column = f"country_{country}"

    if country_column not in model_input.columns:

        available_countries = [
            col.replace("country_", "")
            for col in model_input.columns
            if col.startswith("country_")
        ]

        raise ValueError(
            f"Unknown country: '{country}'.\n\n"
            f"Available countries:\n"
            f"{available_countries}"
        )

    model_input.loc[0, country_column] = 1


    # --------------------------------------------------------
    # FINAL COLUMN CHECK
    # --------------------------------------------------------

    # Ensure exact feature order
    model_input = model_input[FEATURE_NAMES]


    if model_input.shape[1] != len(FEATURE_NAMES):

        raise ValueError(
            f"Feature mismatch!\n"
            f"Expected: {len(FEATURE_NAMES)}\n"
            f"Received: {model_input.shape[1]}"
        )


    # --------------------------------------------------------
    # PREDICT PROBABILITY
    # --------------------------------------------------------

    probability = float(
        final_catboost.predict_proba(
            model_input
        )[0][1]
    )


    # --------------------------------------------------------
    # APPLY SAVED THRESHOLD
    # --------------------------------------------------------

    prediction = int(
        probability >= FINAL_THRESHOLD
    )


    # --------------------------------------------------------
    # DISPLAY RESULT
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("KICKSTARTER SUCCESS PREDICTION")
    print("=" * 60)

    print(f"Project Name     : {name}")
    print(f"Category         : {category}")
    print(f"Main Category    : {main_category}")
    print(f"Country          : {country}")
    print(f"Currency         : {currency}")

    print("-" * 60)

    print(f"Goal             : {goal:g}")
    print(f"USD Goal         : {usd_goal_real:g}")
    print(f"Duration         : {duration_days} days")

    print("-" * 60)

    print(
        f"Success Probability : "
        f"{probability * 100:.2f}%"
    )

    print(
        f"Decision Threshold  : "
        f"{FINAL_THRESHOLD:.2f}"
    )

    if prediction == 1:

        print(
            "Prediction          : SUCCESSFUL"
        )

    else:

        print(
            "Prediction          : NOT SUCCESSFUL"
        )

    print("=" * 60)


    # --------------------------------------------------------
    # RETURN RESULTS
    # --------------------------------------------------------

    return {
        "prediction": prediction,
        "probability": probability,
        "probability_percent": probability * 100,
        "threshold": FINAL_THRESHOLD,
        "features": model_input
    }

FINAL PREDICTION PIPELINE READY
Model features : 219
Threshold      : 0.38
Model          : CatBoostClassifier


In [82]:
result = predict_new_project(
    name="Innovative Smart Home Technology",
    goal=5000,
    usd_goal_real=5000,
    launch_date="2017-04-01",
    deadline_date="2017-06-01",
    category="Technology",
    main_category="Technology",
    currency="USD",
    country="US"
)


KICKSTARTER SUCCESS PREDICTION
Project Name     : Innovative Smart Home Technology
Category         : Technology
Main Category    : Technology
Country          : US
Currency         : USD
------------------------------------------------------------
Goal             : 5000
USD Goal         : 5000
Duration         : 61 days
------------------------------------------------------------
Success Probability : 16.50%
Decision Threshold  : 0.38
Prediction          : NOT SUCCESSFUL


In [83]:
# ============================================================
# FIND A REAL SUCCESSFUL PROJECT WITH HIGH MODEL PROBABILITY
# ============================================================

# Get probabilities for the saved model
test_probabilities = final_catboost.predict_proba(
    X_test
)[:, 1]

# Find actual successful projects
successful_mask = (y_test.values == 1)

successful_indices = np.where(successful_mask)[0]

# Get their probabilities
successful_probs = test_probabilities[successful_indices]

# Find the highest-probability successful project
best_position = np.argmax(successful_probs)
best_index = successful_indices[best_position]

print("=" * 70)
print("BEST REAL SUCCESSFUL PROJECT FROM TEST SET")
print("=" * 70)

print("Test index          :", best_index)
print("Actual target       :", y_test.iloc[best_index])
print(
    "Model probability   :",
    f"{test_probabilities[best_index] * 100:.2f}%"
)
print("Threshold           :", FINAL_THRESHOLD)
print(
    "Model prediction    :",
    "SUCCESSFUL"
    if test_probabilities[best_index] >= FINAL_THRESHOLD
    else "NOT SUCCESSFUL"
)

print("=" * 70)

BEST REAL SUCCESSFUL PROJECT FROM TEST SET
Test index          : 39675
Actual target       : 1
Model probability   : 98.86%
Threshold           : 0.38
Model prediction    : SUCCESSFUL


In [85]:
# ============================================================
# LOAD CLEANED DATASET FOR REAL PROJECT VALIDATION
# ============================================================

import pandas as pd
import os

CLEANED_PATH = (
    "/content/drive/MyDrive/"
    "kickstarter-sucess-prediction (1)/"
    "data/processed/kickstarter_cleaned.csv"
)

cleaned_data = pd.read_csv(CLEANED_PATH)

print("=" * 70)
print("CLEANED DATA LOADED")
print("=" * 70)

print("Shape:", cleaned_data.shape)
print("Columns:")
print(cleaned_data.columns.tolist())

CLEANED DATA LOADED
Shape: (331672, 11)
Columns:
['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'country', 'usd_goal_real', 'target', 'duration_days']


In [86]:
# ============================================================
# CHECK TEST INDEX MAPPING
# ============================================================

test_idx = 39675

print("X_test index at position", test_idx, ":")
print(X_test.index[test_idx])

print("\nCorresponding cleaned-data row:")
print(
    cleaned_data.loc[X_test.index[test_idx]]
)

X_test index at position 39675 :
256428

Corresponding cleaned-data row:
name             Innocents, a truly terrifying roleplaying game
category                                         Tabletop Games
main_category                                             Games
currency                                                    USD
deadline                                             2017-03-13
goal                                                        1.0
launched                                    2017-03-06 19:01:30
country                                                      US
usd_goal_real                                               1.0
target                                                        1
duration_days                                          6.207292
Name: 256428, dtype: object


In [87]:
# ============================================================
# FINAL END-TO-END TEST
# REAL SUCCESSFUL PROJECT
# ============================================================

result = predict_new_project(
    name="Innocents, a truly terrifying roleplaying game",
    goal=1,
    usd_goal_real=1,
    launch_date="2017-03-06",
    deadline_date="2017-03-13",
    category="Tabletop Games",
    main_category="Games",
    currency="USD",
    country="US"
)


KICKSTARTER SUCCESS PREDICTION
Project Name     : Innocents, a truly terrifying roleplaying game
Category         : Tabletop Games
Main Category    : Games
Country          : US
Currency         : USD
------------------------------------------------------------
Goal             : 1
USD Goal         : 1
Duration         : 7 days
------------------------------------------------------------
Success Probability : 98.86%
Decision Threshold  : 0.38
Prediction          : SUCCESSFUL


In [88]:
# FINAL DEPLOYMENT CHECK

print("Model:", type(final_catboost).__name__)
print("Features:", len(feature_names))
print("Threshold:", FINAL_THRESHOLD)

result = predict_new_project(
    name="Innocents, a truly terrifying roleplaying game",
    goal=1,
    usd_goal_real=1,
    launch_date="2017-03-06",
    deadline_date="2017-03-13",
    category="Tabletop Games",
    main_category="Games",
    currency="USD",
    country="US"
)

print("\nFINAL DEPLOYMENT TEST")
print("=" * 50)
print("Probability:", result["probability_percent"], "%")
print("Prediction:", result["prediction"])

Model: CatBoostClassifier
Features: 219
Threshold: 0.38

KICKSTARTER SUCCESS PREDICTION
Project Name     : Innocents, a truly terrifying roleplaying game
Category         : Tabletop Games
Main Category    : Games
Country          : US
Currency         : USD
------------------------------------------------------------
Goal             : 1
USD Goal         : 1
Duration         : 7 days
------------------------------------------------------------
Success Probability : 98.86%
Decision Threshold  : 0.38
Prediction          : SUCCESSFUL

FINAL DEPLOYMENT TEST
Probability: 98.8612738078584 %
Prediction: 1
